# [Kaggle] 06 - Evaluate All Models (Paper Format)

This notebook evaluates the saved models from NB01, NB02, NB04, and NB05 using the artifacts stored in `models/` and `preprocessed/`.
If a model or tokenizer is unavailable in the current environment, the notebook skips that block instead of crashing.


In [ ]:
import json
import pickle
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.jsonl').exists() and (candidate / 'models').exists() and (candidate / 'preprocessed').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root. Expected data/, models/, and preprocessed/ folders.')

ROOT_DIR = find_repo_root()
RESEARCH_DIR = ROOT_DIR / 'research_pipeline'
DATA_DIR = ROOT_DIR / 'data'
MODELS_DIR = ROOT_DIR / 'models'
PREPROCESSED_DIR = ROOT_DIR / 'preprocessed'

if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'ROOT_DIR: {ROOT_DIR}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'MODELS_DIR: {MODELS_DIR}')
print(f'PREPROCESSED_DIR: {PREPROCESSED_DIR}')

ASPECTS = ['CAMERA', 'FEATURES', 'PERFORMANCE', 'DESIGN', 'PRICE', 'GENERAL', 'SCREEN', 'BATTERY', 'STORAGE', 'SER&ACC']
SENTIMENTS = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
LABEL_NAMES = [f'{a}#{s}' for a in ASPECTS for s in SENTIMENTS]

from src.utils.preprocess import load_raw_data, tokenize_baseline
from src.utils.metrics import bio_tags_to_spans, evaluate_spans_f1, evaluate_asc
from src.utils.engine import predict_ate, predict_asc
from src.ate.ate_dataset import ATEDataset, BIO_TAGS as BIO_TAGS_ATE
from src.ate.ate_model import build_ate_model
from src.asc.asc_dataset import ASCDataset
from src.asc.asc_model import build_asc_model
from src.e2e.e2e_baseline_dataset import E2EBaselineDataset, BIO_TAGS as BIO_TAGS_E2E_BASE
from src.e2e.e2e_dataset import E2EDataset, BIO_TAGS as BIO_TAGS_E2E, NUM_TAGS as NUM_TAGS_E2E
from src.e2e.e2e_model import E2EPhoBertCRF

with open(PREPROCESSED_DIR / 'word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)
emb_matrix = np.load(PREPROCESSED_DIR / 'emb_matrix.npy')

with open(PREPROCESSED_DIR / 'test_segmented.json', 'r', encoding='utf-8') as f:
    test_items_seg = json.load(f)

test_items_raw = load_raw_data(DATA_DIR / 'test.jsonl')

print(f'Loaded vocab: {len(word2idx)}')
print(f'Embeddings: {emb_matrix.shape}')
print(f'Test items: raw={len(test_items_raw)} segmented={len(test_items_seg)}')

MAX_LEN_ATE = 128
MAX_LEN_ASC = 128
MAX_LEN_PHOBERT = 256

def load_state_dict_if_exists(model, path: Path):
    if not path.exists():
        print(f'  Missing model file: {path}')
        return None
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

def evaluate_absa_paper_format(pred_spans_list, true_spans_list, model_name='Model'):
    metrics = {'Aspect': {'tp': 0, 'fp': 0, 'fn': 0}, 'Polarity': {'tp': 0, 'fp': 0, 'fn': 0}, 'Aspect-Polarity': {'tp': 0, 'fp': 0, 'fn': 0}}
    aspect_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    aspect_sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for true_spans, pred_spans in zip(true_spans_list, pred_spans_list):
        true_ap = set((label, s, e) for label, s, e in true_spans)
        pred_ap = set((label, s, e) for label, s, e in pred_spans)
        true_a = set((label.split('#')[0], s, e) for label, s, e in true_spans)
        pred_a = set((label.split('#')[0], s, e) for label, s, e in pred_spans)
        true_p = set(((label.split('#')[1] if '#' in label else 'UNK'), s, e) for label, s, e in true_spans)
        pred_p = set(((label.split('#')[1] if '#' in label else 'UNK'), s, e) for label, s, e in pred_spans)

        def update_counts(true_set, pred_set, overall_dict, class_dict):
            for span in true_set:
                key = span[0]
                if span in pred_set:
                    overall_dict['tp'] += 1
                    class_dict[key]['tp'] += 1
                else:
                    overall_dict['fn'] += 1
                    class_dict[key]['fn'] += 1
            for span in pred_set:
                if span not in true_set:
                    key = span[0]
                    overall_dict['fp'] += 1
                    class_dict[key]['fp'] += 1

        update_counts(true_ap, pred_ap, metrics['Aspect-Polarity'], aspect_sentiment_counts)
        update_counts(true_a, pred_a, metrics['Aspect'], aspect_counts)
        update_counts(true_p, pred_p, metrics['Polarity'], sentiment_counts)

    def calc_metrics(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp > 0 else 0.0
        r = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0.0
        return p * 100, r * 100, f1 * 100

    def calc_macro(counts_dict, keys):
        values = [calc_metrics(counts_dict[k]['tp'], counts_dict[k]['fp'], counts_dict[k]['fn']) for k in keys]
        return tuple(np.mean([v[i] for v in values]) for i in range(3))

    rows = []
    for task in ['Aspect', 'Polarity', 'Aspect-Polarity']:
        c = metrics[task]
        p_micro, r_micro, f1_micro = calc_metrics(c['tp'], c['fp'], c['fn'])
        if task == 'Aspect':
            keys = ASPECTS
            counts = aspect_counts
        elif task == 'Polarity':
            keys = SENTIMENTS
            counts = sentiment_counts
        else:
            keys = [f'{a}#{s}' for a in ASPECTS for s in SENTIMENTS]
            counts = aspect_sentiment_counts
        p_macro, r_macro, f1_macro = calc_macro(counts, keys)
        rows.append({'System': task, 'PMicro': p_micro, 'RMicro': r_micro, 'F1Micro': f1_micro, 'PMacro': p_macro, 'RMacro': r_macro, 'F1Macro': f1_macro})

    df = pd.DataFrame(rows).round(2).set_index('System')
    print(f'\n{"=" * 90}\nPaper-format evaluation: {model_name}\n{"=" * 90}')
    display(df)
    return df

def evaluate_ate_spans(pred_spans_list, true_spans_list, model_name='Model'):
    res = evaluate_spans_f1(pred_spans_list, true_spans_list)
    df = pd.DataFrame([{'Model': model_name, 'Precision': res['precision'] * 100, 'Recall': res['recall'] * 100, 'F1': res['f1'] * 100}]).round(2).set_index('Model')
    print(f'\n{"=" * 90}\nATE evaluation: {model_name}\n{"=" * 90}')
    display(df)
    return df

def evaluate_asc_predictions(y_true, y_pred, model_name='Model'):
    res = evaluate_asc(y_true, y_pred, class_names=['POSITIVE', 'NEGATIVE', 'NEUTRAL'])
    df = pd.DataFrame([{'Model': model_name, 'Accuracy': res['accuracy'] * 100, 'Macro_F1': res['macro_f1'] * 100, 'Weighted_F1': res['weighted_f1'] * 100}]).round(2).set_index('Model')
    print(f'\n{"=" * 90}\nASC evaluation: {model_name}\n{"=" * 90}')
    display(df)
    return df

def segmented_raw_labels_to_word_spans(seg_text, raw_labels, max_len):
    words = seg_text.split()[:max_len]
    positions = []
    pos = 0
    for w in words:
        w_len = len(w.replace('_', ' '))
        positions.append((pos, pos + w_len))
        pos += w_len + 1
    spans = []
    for start_char, end_char, label in raw_labels:
        word_indices = []
        for idx, (w_start, w_end) in enumerate(positions):
            if w_start < end_char and w_end > start_char:
                word_indices.append(idx)
        if word_indices:
            spans.append((label, word_indices[0], word_indices[-1] + 1))
    return spans

def pipeline_predict(seg_text, ate_model, asc_model, word2idx, device, max_len=128):
    words = seg_text.split()[:max_len]
    if not words:
        return []

    seq, length = tokenize_baseline(seg_text, word2idx, max_len)
    seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)
    len_tensor = torch.tensor([length], dtype=torch.long)
    mask = torch.zeros((1, max_len), dtype=torch.bool, device=device)
    mask[:, :length] = True

    with torch.no_grad():
        pred_tags = ate_model(seq_tensor, mask=mask, lens=len_tensor)[0]

    aspect_spans = bio_tags_to_spans(pred_tags, BIO_TAGS_ATE, length)
    results = []

    for aspect, start, end in aspect_spans:
        marked_words = words[:start] + ['[ASP]'] + words[start:end] + ['[ASP]'] + words[end:]
        marked_text = ' '.join(marked_words)
        asc_seq, asc_len = tokenize_baseline(marked_text, word2idx, max_len)
        asc_seq_tensor = torch.tensor([asc_seq], dtype=torch.long, device=device)
        asc_len_tensor = torch.tensor([asc_len], dtype=torch.long)
        with torch.no_grad():
            logits = asc_model(asc_seq_tensor, asc_len_tensor)
            sentiment_id = logits.argmax(dim=1).item()
        sentiment = {0: 'POSITIVE', 1: 'NEGATIVE', 2: 'NEUTRAL'}[sentiment_id]
        results.append((f'{aspect}#{sentiment}', start, end))

    return results

results_absa = []
results_ate = []
results_asc = []

print('\nPreparing datasets...')
test_ds_bigru = E2EBaselineDataset(test_items_seg, word2idx, MAX_LEN_ATE)
test_ds_ate = ATEDataset(test_items_seg, word2idx, MAX_LEN_ATE)
test_ds_asc = ASCDataset(test_items_seg, word2idx, MAX_LEN_ASC)
test_loader_bigru = DataLoader(test_ds_bigru, batch_size=64, shuffle=False)
test_loader_ate = DataLoader(test_ds_ate, batch_size=64, shuffle=False)
test_loader_asc = DataLoader(test_ds_asc, batch_size=128, shuffle=False)
print(f'BiGRU baseline samples: {len(test_ds_bigru)}')
print(f'ATE samples: {len(test_ds_ate)}')
print(f'ASC samples: {len(test_ds_asc)}')

print('\nEvaluating E2E models...')
try:
    test_ds_phobert = E2EDataset(test_items_raw, max_len=MAX_LEN_PHOBERT)
    test_loader_phobert = DataLoader(test_ds_phobert, batch_size=16, shuffle=False)
    phobert_path = MODELS_DIR / 'best_e2e_phobert.pt'
    phobert_model = E2EPhoBertCRF(num_unified_tags=NUM_TAGS_E2E, dropout=0.3)
    if load_state_dict_if_exists(phobert_model, phobert_path) is not None:
        from src.e2e.e2e_engine import predict_e2e
        test_res_phobert = predict_e2e(phobert_model, test_loader_phobert, device)
        pred_spans_phobert = [bio_tags_to_spans(pt, BIO_TAGS_E2E, l) for pt, l in zip(test_res_phobert['pred_tags'], test_res_phobert['lengths'])]
        true_spans_phobert = [bio_tags_to_spans(tt, BIO_TAGS_E2E, l) for tt, l in zip(test_res_phobert['true_tags'], test_res_phobert['lengths'])]
        df = evaluate_absa_paper_format(pred_spans_phobert, true_spans_phobert, model_name='E2E PhoBERT-CRF')
        results_absa.append(('E2E PhoBERT-CRF', df))
except Exception as exc:
    print(f'  Skipping E2E PhoBERT-CRF: {exc}')

try:
    e2e_bigru_path = MODELS_DIR / 'best_e2e_baseline_BiGRU-CRF.pt'
    e2e_bigru_model = build_ate_model(model_type='BiGRU', vocab_size=len(word2idx), emb_dim=150, hidden_dim=256, num_tags=len(BIO_TAGS_E2E_BASE), pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
    if load_state_dict_if_exists(e2e_bigru_model, e2e_bigru_path) is not None:
        test_res_bigru = predict_ate(e2e_bigru_model, test_loader_bigru, device, bio_tags_list=BIO_TAGS_E2E_BASE)
        pred_spans_bigru = [bio_tags_to_spans(pt, BIO_TAGS_E2E_BASE, l) for pt, l in zip(test_res_bigru['pred_tags'], test_res_bigru['lengths'])]
        true_spans_bigru = [bio_tags_to_spans(tt, BIO_TAGS_E2E_BASE, l) for tt, l in zip(test_res_bigru['true_tags'], test_res_bigru['lengths'])]
        df = evaluate_absa_paper_format(pred_spans_bigru, true_spans_bigru, model_name='E2E BiGRU-CRF')
        results_absa.append(('E2E BiGRU-CRF', df))
except Exception as exc:
    print(f'  Skipping E2E BiGRU-CRF: {exc}')

print('\nEvaluating standalone ATE model...')
try:
    ate_path = MODELS_DIR / 'best_ate_BiLSTM-CRF.pt'
    ate_model = build_ate_model(model_type='BiLSTM', vocab_size=len(word2idx), emb_dim=150, hidden_dim=256, num_tags=len(BIO_TAGS_ATE), pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
    if load_state_dict_if_exists(ate_model, ate_path) is not None:
        test_res_ate = predict_ate(ate_model, test_loader_ate, device, bio_tags_list=BIO_TAGS_ATE)
        pred_spans_ate = [bio_tags_to_spans(pt, BIO_TAGS_ATE, l) for pt, l in zip(test_res_ate['pred_tags'], test_res_ate['lengths'])]
        true_spans_ate = [bio_tags_to_spans(tt, BIO_TAGS_ATE, l) for tt, l in zip(test_res_ate['true_tags'], test_res_ate['lengths'])]
        df = evaluate_ate_spans(pred_spans_ate, true_spans_ate, model_name='ATE BiLSTM-CRF')
        results_ate.append(('ATE BiLSTM-CRF', df))
except Exception as exc:
    print(f'  Skipping ATE BiLSTM-CRF: {exc}')

print('\nEvaluating standalone ASC model...')
try:
    asc_path = MODELS_DIR / 'best_asc_BiGRU.pt'
    asc_model = build_asc_model(model_type='BiGRU', vocab_size=len(word2idx), emb_dim=150, hidden_dim=256, num_classes=3, pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
    if load_state_dict_if_exists(asc_model, asc_path) is not None:
        criterion = torch.nn.CrossEntropyLoss()
        test_res_asc = predict_asc(asc_model, test_loader_asc, criterion, device)
        df = evaluate_asc_predictions(test_res_asc['labels'], test_res_asc['preds'], model_name='ASC BiGRU')
        results_asc.append(('ASC BiGRU', df))
except Exception as exc:
    print(f'  Skipping ASC BiGRU: {exc}')

print('\nEvaluating pipeline model (ATE + ASC)...')
try:
    pred_spans_pipe = []
    true_spans_pipe = []
    for raw_item, seg_item in zip(test_items_raw, test_items_seg):
        pred_spans_pipe.append(pipeline_predict(seg_item['text'], ate_model, asc_model, word2idx, device, max_len=MAX_LEN_ASC))
        true_spans_pipe.append(segmented_raw_labels_to_word_spans(seg_item['text'], raw_item['labels'], MAX_LEN_ASC))
    df = evaluate_absa_paper_format(pred_spans_pipe, true_spans_pipe, model_name='Pipeline (ATE BiLSTM + ASC BiGRU)')
    results_absa.append(('Pipeline (ATE BiLSTM + ASC BiGRU)', df))
except Exception as exc:
    print(f'  Skipping pipeline evaluation: {exc}')

print('\nSummary')
summary_rows = []
for model_name, df in results_absa:
    if 'Aspect-Polarity' in df.index:
        row = df.loc['Aspect-Polarity']
        summary_rows.append({'Model': model_name, 'Aspect-Polarity F1Micro': row['F1Micro'], 'Aspect-Polarity F1Macro': row['F1Macro']})
for model_name, df in results_ate:
    row = df.iloc[0]
    summary_rows.append({'Model': model_name, 'ATE F1': row['F1']})
for model_name, df in results_asc:
    row = df.iloc[0]
    summary_rows.append({'Model': model_name, 'ASC Accuracy': row['Accuracy'], 'ASC Macro F1': row['Macro_F1']})

summary_df = pd.DataFrame(summary_rows).round(2)
if not summary_df.empty:
    display(summary_df)
else:
    print('No results were produced.')
